# Synthetic spatial showcase: points, polygons, distance and semantic maps
Twelve fictional service sites and four rectangular demonstration zones near a configurable map centre. These are NOT municipal assets, administrative boundaries, police locations or operational statistics. EPSG:4326; WKT/GeoJSON coordinate order is longitude, latitude. Latitude/longitude columns are separate for Azure Maps. No GIS libraries or external downloads are needed.

The rectangular spatial join below is exact only for these axis-aligned demo polygons. For real polygons use a supported spatial engine with point-in-polygon and explicit CRS handling, not the bounding-box shortcut.


In [ ]:
center_latitude = 44.65
center_longitude = -63.58
cell_size_degrees = 0.04
sites_table = 'dbo.gold_spatial_sites'
zones_table = 'dbo.gold_spatial_zones'


In [ ]:
import json
import math
import re
from pyspark.sql import functions as F

for table in (sites_table, zones_table):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*', table):
        raise ValueError('Use schema.table identifiers only.')
if not (0 < cell_size_degrees <= 1 and -89 < center_latitude < 89 and -178 < center_longitude < 178):
    raise ValueError('Choose a small positive cell size and centre away from poles/date line.')
zone_rows, features, site_rows = [], [], []
for row in range(2):
    for col in range(2):
        zone_index = row * 2 + col + 1
        zone_id = f'Z{zone_index:02d}'
        west = center_longitude + (col - 1) * cell_size_degrees
        south = center_latitude + (row - 1) * cell_size_degrees
        east, north = west + cell_size_degrees, south + cell_size_degrees
        ring = [[west, south], [east, south], [east, north], [west, north], [west, south]]
        wkt = 'POLYGON ((' + ', '.join(f'{x} {y}' for x, y in ring) + '))'
        zone_rows.append((zone_id, f'Demo Zone {zone_index}', west, south, east, north, wkt))
        features.append({'type': 'Feature', 'id': zone_id, 'properties': {'zone_id': zone_id, 'zone_name': f'Demo Zone {zone_index}'}, 'geometry': {'type': 'Polygon', 'coordinates': [ring]}})
        for i, (dx, dy) in enumerate([(0.2, 0.25), (0.55, 0.65), (0.8, 0.4)]):
            site_number = (zone_index - 1) * 3 + i + 1
            lon, lat = west + dx * cell_size_degrees, south + dy * cell_size_degrees
            site_rows.append((f'S{site_number:03d}', f'Synthetic Site {site_number}', ['Park', 'Community Centre', 'Service Desk'][i], lat, lon, int(5 + (site_number * 7) % 40), f'POINT ({lon} {lat})'))
zones = spark.createDataFrame(zone_rows, 'zone_id string, zone_name string, west double, south double, east double, north double, geometry_wkt string')
sites = spark.createDataFrame(site_rows, 'site_id string, site_name string, category string, latitude double, longitude double, service_requests long, geometry_wkt string')
joined = (sites.alias('s').join(F.broadcast(zones.alias('z')),
    (F.col('s.longitude') >= F.col('z.west')) & (F.col('s.longitude') < F.col('z.east')) &
    (F.col('s.latitude') >= F.col('z.south')) & (F.col('s.latitude') < F.col('z.north')), 'left')
    .select('s.*', 'z.zone_id'))
dlat = F.radians(F.col('latitude') - F.lit(center_latitude))
dlon = F.radians(F.col('longitude') - F.lit(center_longitude))
a = F.pow(F.sin(dlat / 2), 2) + F.cos(F.radians(F.col('latitude'))) * F.lit(math.cos(math.radians(center_latitude))) * F.pow(F.sin(dlon / 2), 2)
gold = joined.withColumn('distance_km', F.lit(6371.0088 * 2) * F.asin(F.sqrt(F.least(F.lit(1.0), F.greatest(F.lit(0.0), a)))))
assert gold.count() == 12 and gold.filter('zone_id IS NULL').count() == 0
# Deterministic overwrite makes this synthetic data generation safe to replay.
gold.write.format('delta').mode('overwrite').saveAsTable(sites_table)
zones.write.format('delta').mode('overwrite').saveAsTable(zones_table)
notebookutils.fs.mkdirs('Files/geo')
notebookutils.fs.put('Files/geo/showcase_zones.geojson', json.dumps({'type': 'FeatureCollection', 'features': features}), True)
display(gold.orderBy('site_id'))
display(gold.groupBy('zone_id').agg(F.count('*').alias('sites'), F.sum('service_requests').alias('requests')))
